# 🤖 AI-Powered Study Assistant
### RAG + Multi-Agent System | LangChain · CrewAI · Groq · ChromaDB

## About
This project demonstrates a **RAG pipeline (Retrieval-Augmented Generation)** 
combined with a **Multi-Agent System (Agent-to-Agent Communication)**.

Real university lecture slides (344 pages, 7 PDFs) serve as the knowledge base.
Two specialized AI agents collaborate:
- **Agent 1 (Classifier)** – classifies the query by topic and complexity
- **Agent 2 (Solver)** – searches the vector database and formulates an answer

## Tech Stack
| Component | Technology |
|---|---|
| LLM | Llama 4 via Groq API |
| RAG Framework | LangChain |
| Agent Framework | CrewAI |
| Vector Database | ChromaDB |
| Embeddings | HuggingFace (all-MiniLM-L6-v2) |
| Environment | Kaggle Notebook |


# ── Cell 1: Install Dependencies ────────────────────────────────────────


In [1]:
import subprocess, sys

# Install all required packages with pinned versions for compatibility
# stderr=DEVNULL suppresses pip dependency warnings (cosmetic only)
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "langchain==0.3.25",
    "langchain-core==0.3.83",        # pinned: newer versions break compatibility
    "langchain-community==0.3.24",
    "langchain-groq==0.3.2",
    "langchain-huggingface==0.1.2",
    "chromadb==0.6.3",
    "crewai==0.114.0",
    "pypdf==5.4.0",
    "sentence-transformers",
    "--upgrade"
], stderr=subprocess.DEVNULL)

# Verify installed versions
import langchain, langchain_core, langchain_groq
print(f"✅ langchain: {langchain.__version__}")
print(f"✅ langchain-core: {langchain_core.__version__}")
print(f"✅ langchain-groq: {langchain_groq.__version__}")
print("🎉 All packages ready!")


✅ langchain: 0.3.25
✅ langchain-core: 0.3.83
✅ langchain-groq: 0.3.2
🎉 All packages ready!


# ── Cell 2: API Setup & Model Configuration ───────────────────────


In [2]:
import os, warnings, logging
from kaggle_secrets import UserSecretsClient
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

# Load API key securely from Kaggle Secrets (never hardcode API keys!)
os.environ["GROQ_API_KEY"] = UserSecretsClient().get_secret("GROQ_API_KEY")

# LLM: Llama 4 via Groq API – used for both RAG answers and agent reasoning
# temperature=0.0 → deterministic, factual responses (no creativity)
llm = ChatGroq(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    temperature=0.0,
)

# Embeddings: local HuggingFace model – no API call needed, runs on Kaggle GPU
# all-MiniLM-L6-v2 is optimized for semantic similarity search
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

warnings.resetwarnings()

# Quick connectivity test
response = llm.invoke("Say hello in one word.")
print(f"✅ Groq API connected! Response: {response.content}")
print("✅ Embedding model loaded!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Groq API connected! Response: Hello!
✅ Embedding model loaded!


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


# ── Cell 3: Locate PDF Files ─────────────────────────────────────────


In [3]:
import glob, os

# Path to uploaded PDF dataset in Kaggle
# To use your own PDFs: upload via "Add Data → Upload" and adjust this path
base_path = "/kaggle/input/datasets/divinefirstyolo/folien-fh/"
all_pdfs = glob.glob(base_path + "*.pdf")

print(f"📂 Found PDFs ({len(all_pdfs)} files):")
for pdf in all_pdfs:
    print(f"  → {pdf.split('/')[-1]}")

📂 Found PDFs (7 files):
  → AI25 Part1 AI Motivation and Definition.pdf
  → AI25 Part2 Cloud AI Computer Vision.pdf
  → 2025_10_FolienLehrveranstaltung.pdf
  → Methoden des Technologiemanagements.pdf
  → AI25 Part3 LLM genKI.pdf
  → LV_IT-Security_1_2024.pdf
  → LV_IT-Security_2_2024.pdf


# ── Cell 4: Build RAG Pipeline & Vector Database ────────────


In [4]:
import warnings, logging
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

# Suppress ChromaDB telemetry warnings (known bug in chromadb 0.6.3)
os.environ["ANONYMIZED_TELEMETRY"] = "False"
logging.getLogger("chromadb").setLevel(logging.ERROR)

# Step 1: Load all PDFs page by page
all_docs = []
for path in all_pdfs:
    try:
        loader = PyPDFLoader(path)
        all_docs.extend(loader.load())
        print(f"✅ {path.split('/')[-1]}")
    except Exception as e:
        print(f"⚠️ Skipped: {path.split('/')[-1]} → {e}")

print(f"\n📚 {len(all_docs)} pages loaded")

# Step 2: Split documents into chunks for semantic search
# chunk_size=800: balanced between context and precision
# chunk_overlap=100: prevents context loss at chunk boundaries
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = splitter.split_documents(all_docs)
print(f"✂️  {len(chunks)} chunks created")

# Step 3: Embed chunks and store in ChromaDB vector database
# Each chunk is converted to a vector → enables semantic similarity search
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
print("✅ Vector database ready!")

✅ AI25 Part1 AI Motivation and Definition.pdf
✅ AI25 Part2 Cloud AI Computer Vision.pdf
✅ 2025_10_FolienLehrveranstaltung.pdf
✅ Methoden des Technologiemanagements.pdf
✅ AI25 Part3 LLM genKI.pdf
✅ LV_IT-Security_1_2024.pdf
✅ LV_IT-Security_2_2024.pdf

📚 344 pages loaded
✂️  412 chunks created


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/chromadb/types.py:144: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  return self.model_fields  # pydantic 2.x
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ Vector database ready!


# ── Cell 5: RAG Test (Single Query) ──────────────────────────────────────────


In [5]:
from langchain.chains import RetrievalQA

# Build QA chain: retriever fetches relevant chunks → LLM formulates answer
# chain_type="stuff": all retrieved chunks are passed together to the LLM
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
)

# Test query – change this to any topic covered in your PDFs
query = "What is a Large Language Model?"

result = qa_chain.invoke({"query": query})

print("🧠 ANSWER:")
print(result["result"])
print("\n📎 SOURCES:")
for doc in result["source_documents"]:
    print(f"  → Page {doc.metadata.get('page', '?')} | {doc.metadata.get('source', '?').split('/')[-1]}")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/chromadb/types.py:144: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  return self.model_fields  # pydantic 2.x
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datet

🧠 ANSWER:
Based on the provided context, a Large Language Model appears to be a type of Foundation Model (FM), which is a powerful machine learning model that can be pretrained on vast amounts of data to perform a wide range of downstream tasks, including text generation, data summarization, information extraction, question and answer responses, and chatbot interactions.

Specifically, Large Language Models are likely characterized by their large number of parameters, which can range from hundreds of millions to billions. Examples of Large Language Models mentioned in the context include:

* GPT-1 with 117 million parameters
* GPT-3 with 175 billion parameters
* GPT-4, a multimodal model that can analyze text or images
* BLOOM, a multilingual model that can create text in 46 languages and write code in 13 programming languages
* PaLM, a 540-billion-parameter model developed by Google
* Mistral 7B, a 7-billion-parameter model that excels in reasoning and mathematics

These models are tr

# ── Cell 6: Multi-Agent Chat UI (CrewAI + ipywidgets) ──────────────────


In [6]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from crewai import Agent, Task, Crew, Process
from langchain_groq import ChatGroq

# "groq/" prefix required so LiteLLM (used internally by CrewAI) recognizes the provider
CREWAI_MODEL = "groq/meta-llama/llama-4-scout-17b-16e-instruct"
translation_llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct", temperature=0)

# ── UI Components ─────────────────────────────────────────────────────────────
title = widgets.HTML("""
<div style='font-family: Arial, sans-serif; padding: 16px 0'>
    <h2 style='margin:0; color:#111'>🤖 AI Study Assistant</h2>
    <p style='color:#555; margin:4px 0'>Ask questions in German or English – answers come exclusively from your uploaded PDFs.</p>
</div>
""")

input_box = widgets.Textarea(
    placeholder="Enter your question (German or English)...",
    layout=widgets.Layout(width="100%", height="80px")
)

run_button = widgets.Button(
    description="Start Agents →",
    button_style="primary",
    layout=widgets.Layout(width="180px", height="36px"),
)

output = widgets.Output()

def on_button_click(b):
    query = input_box.value.strip()
    if not query:
        with output:
            clear_output()
            display(HTML("<p style='color:red; font-family:Arial'>⚠️ Please enter a question.</p>"))
        return

    with output:
        clear_output()

        # ── Agent 1: Query Classification ─────────────────────────
        display(HTML("""
        <div style='font-family:Arial; background:#f5f5f5; border:1px solid #ddd;
                    border-left:4px solid #555; padding:12px; border-radius:4px; margin:8px 0'>
            <b style='color:#333'>🔍 Agent 1: Classifier</b>
            <span style='color:#888; font-size:13px'>&nbsp;⏳ analyzing...</span>
        </div>
        """))

        # Agent 1: classifies topic, complexity and keywords
        classifier = Agent(
            role="Query Classifier",
            goal="Classify the query by topic and complexity.",
            backstory="Reply ONLY in this format: TOPIC: X | COMPLEXITY: Y | KEYWORDS: Z",
            llm=CREWAI_MODEL, verbose=False,
        )
        task1 = Task(
            description=f'Classify:\n"{query}"',
            expected_output="TOPIC: X | COMPLEXITY: Y | KEYWORDS: Z",
            agent=classifier,
        )
        result1 = Crew(agents=[classifier], tasks=[task1], verbose=False).kickoff()
        classification = str(result1)

        display(HTML(f"""
        <div style='font-family:Arial; background:#f5f5f5; border:1px solid #ddd;
                    border-left:4px solid #555; padding:12px; border-radius:4px; margin:8px 0'>
            <b style='color:#333'>🔍 Agent 1: Classifier</b>
            <span style='color:#2a7a2a; font-size:13px'>&nbsp;✅ done</span><br><br>
            <code style='background:#e8e8e8; padding:8px; border-radius:4px; display:block; color:#222; font-size:13px'>
                {classification}
            </code>
        </div>
        """))

        # ── Translation for RAG Search ────────────────────────────
        # PDFs are in English → translate query before vector search
        display(HTML("""
        <div style='font-family:Arial; background:#fafafa; border:1px solid #ddd;
                    border-left:4px solid #222; padding:12px; border-radius:4px; margin:8px 0'>
            <b style='color:#333'>📚 Agent 2: Knowledge Assistant</b>
            <span style='color:#888; font-size:13px'>&nbsp;⏳ searching PDFs...</span>
        </div>
        """))

        translation = translation_llm.invoke(
            f"Translate this question to English. Return ONLY the translated question, nothing else: {query}"
        )
        query_english = translation.content.strip()

        # ── RAG: Retrieve relevant chunks using English query ─────
        extended_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
        relevant_docs = extended_retriever.invoke(query_english)
        context = "\n\n---\n\n".join([doc.page_content for doc in relevant_docs])

        # ── Agent 2: Answer Generation ────────────────────────────
        # Solver uses ONLY the retrieved chunks – no internet, no hallucination
        solver = Agent(
            role="Knowledge Assistant",
            goal="Answer questions ONLY based on the provided text excerpts.",
            backstory="""You MUST use ONLY the text excerpts provided below.
            You have NO internet access. Do NOT invent sources.
            If the answer is not in the excerpts, say:
            'This information is not available in the provided documents.'""",
            llm=CREWAI_MODEL, verbose=False,
        )
        task2 = Task(
            description=f"""Answer in German: "{query}"
English search query used: "{query_english}"

AVAILABLE TEXT EXCERPTS FROM DOCUMENTS:
{context}

IMPORTANT: Use ONLY these excerpts. No internet, no external sources.""",
            expected_output="Structured answer in German, based ONLY on the provided excerpts.",
            agent=solver,
        )
        result2 = Crew(agents=[solver], tasks=[task2], verbose=False).kickoff()
        answer = str(result2)

        sources_html = "".join([
            f"<li style='margin:4px 0'>📄 Page {d.metadata.get('page','?')} &nbsp;|&nbsp; "
            f"<b>{d.metadata.get('source','?').split('/')[-1]}</b></li>"
            for d in relevant_docs
        ])

        display(HTML(f"""
        <div style='font-family:Arial; background:#fafafa; border:1px solid #ddd;
                    border-left:4px solid #222; padding:12px; border-radius:4px; margin:8px 0'>
            <b style='color:#333'>📚 Agent 2: Knowledge Assistant</b>
            <span style='color:#2a7a2a; font-size:13px'>&nbsp;✅ done</span><br><br>
            <p style='color:#222; line-height:1.6'>{answer}</p>
        </div>
        <div style='font-family:Arial; background:#fff; border:1px solid #ddd;
                    padding:12px; border-radius:4px; margin:8px 0'>
            <b style='color:#333'>📎 Sources used (from your PDFs):</b>
            <ul style='color:#444; margin:8px 0'>{sources_html}</ul>
            <p style='color:#888; font-size:12px; margin:4px 0'>
                ℹ️ No internet access – all answers are based exclusively on the uploaded PDFs.
                &nbsp;|&nbsp; Internal search query: <i>"{query_english}"</i>
            </p>
        </div>
        """))

run_button.on_click(on_button_click)
display(title, input_box, run_button, output)

/usr/local/lib/python3.12/dist-packages/litellm/types/llms/openai.py:624: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class LiteLLMFineTuningJobCreate(FineTuningJobCreate):
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc

HTML(value="\n<div style='font-family: Arial, sans-serif; padding: 16px 0'>\n    <h2 style='margin:0; color:#1…

Textarea(value='', layout=Layout(height='80px', width='100%'), placeholder='Enter your question (German or Eng…

Button(button_style='primary', description='Start Agents →', layout=Layout(height='36px', width='180px'), styl…

Output()